# MatMul 量化模式与实现介绍

随着模型规模持续增长，全精度浮点运算在带宽、存储和算力方面面临显著压力。本节系统介绍 MatMul 算子量化的基本原理、五种量化粒度及 MX 量化实现方案，建立「量化模式 → Scale 形状 → 实现选型」的认知体系，为后续 [01.07 基于 Blaze 的矩阵算子开发](./01.07_blaze_based_matmul_operator_development.ipynb) 的实操奠定基础。

本节学习大纲如下：

- 理解量化的动机与基本定义
- 区分静态量化与动态量化，掌握适用场景
- 掌握五种量化粒度（T/C/K/G/B）及 Scale Shape 规律
- 了解常见组合量化模式
- 了解 MX 量化实现原理

---

## 1. CANN 算子量化基础

### 1.1 量化的动机

随着大语言模型参数量达到千亿乃至万亿级别，全精度运算带来三方面挑战：

- **存储压力**：权重矩阵占用大量显存，限制模型部署规模
- **带宽瓶颈**：数据搬运速度往往成为算子性能的主要制约因素
- **算力受限**：全精度运算难以充分发挥专用矩阵单元针对低 bit 数据的峰值算力


### 1.2 MatMul 量化定义

**MatMul 量化**是指对矩阵（Cube）类算子输入 Tensor 执行从高 bit 到低 bit 转换的计算过程，同时生成对应的量化参数 **Scale**。量化后的低 bit 数据送入 Cube 单元完成矩阵乘计算，计算结果再通过 Scale 反量化回目标精度，从而保证与全精度计算近似等价。

整个计算流程包含三个阶段：

1. **量化（Quantize）**：将输入矩阵 A、B 从高 bit 浮点转换为低 bit 表示，并计算对应的 Scale 张量
2. **低 bit 矩阵乘（Cube Compute）**：在 AI Core Cube 单元上执行低 bit 矩阵乘法
3. **反量化（Dequantize）**：利用 Scale 将低 bit 计算结果恢复为目标精度输出


计算流程可概括为：

```
C[M,N] = dequant( quant(A[M,K]) × quant(B[K,N]) )
```

其中 `quant()` 表示量化操作，`dequant()` 表示反量化操作。Scale 张量的形状由量化粒度决定，直接影响算子实现的 Tiling 策略与 Kernel 选型。

### 1.3 Scale 的作用

Scale 是量化过程中的核心参数，用于记录每个量化分组的数据缩放因子。在矩阵乘场景中，左矩阵 A 和右矩阵 B 各自拥有独立的 Scale 张量（ScaleA 和 ScaleB）。反量化时，对应分组的 Scale 参与累加计算，将低 bit 乘积结果恢复为全精度输出。

---

## 2. 静态量化 vs 动态量化

量化过程中 **Scale 的获取时机** 是区分静态量化与动态量化的核心依据。

**静态量化（Static Quantization）** 指 Scale 在推理前通过离线校准（如使用代表性数据集统计数值范围）预先确定，推理阶段直接复用，无需额外计算。由于 Scale 固定不变，静态量化通常应用于数值分布相对稳定的 **权重（weight）** 矩阵。

**动态量化（Dynamic Quantization）** 指 Scale 在推理过程中根据当前输入数据在线计算，能够自适应每一批次的实际数值分布。动态量化更常用于分布随输入变化的 **激活（activation）** 矩阵，精度适应性更好，但会引入额外的在线 Scale 计算开销。

二者对比如下：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">对比项</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">静态量化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">动态量化</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">Scale 来源</td>
      <td style="text-align: left; padding: 6px 12px;">预先确定，离线校准</td>
      <td style="text-align: left; padding: 6px 12px;">在线根据输入数据计算</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">典型对象</td>
      <td style="text-align: left; padding: 6px 12px;">权重 weight</td>
      <td style="text-align: left; padding: 6px 12px;">激活 activation</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">性能表现</td>
      <td style="text-align: left; padding: 6px 12px;">更好，无在线 Scale 计算开销</td>
      <td style="text-align: left; padding: 6px 12px;">略差</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">精度适应性</td>
      <td style="text-align: left; padding: 6px 12px;">固定 Scale，对分布变化敏感</td>
      <td style="text-align: left; padding: 6px 12px;">更适应数据分布变化</td>
    </tr>
  </tbody>
</table>
</div>

---

## 3. 五种量化粒度

量化粒度（Quantization Granularity）决定了 Tensor 中哪些元素共享同一个 Scale 参数。粒度越细，Scale 数量越多，量化精度越高，但存储与计算开销也相应增加。假设左矩阵 shape 为 **(m, k)**，右矩阵 shape 为 **(k, n)**，**k 为 reduce 轴**（内积求和方向）。

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">模式</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">简称</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">量化对象</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Scale Shape</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">pertensor</td>
      <td style="text-align: left; padding: 6px 12px;">T</td>
      <td style="text-align: left; padding: 6px 12px;">左或右矩阵，整 Tensor 共用</td>
      <td style="text-align: left; padding: 6px 12px;">(1,)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">perchannel</td>
      <td style="text-align: left; padding: 6px 12px;">C</td>
      <td style="text-align: left; padding: 6px 12px;">右矩阵，每 channel 独立</td>
      <td style="text-align: left; padding: 6px 12px;">(n,)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">pertoken</td>
      <td style="text-align: left; padding: 6px 12px;">K</td>
      <td style="text-align: left; padding: 6px 12px;">左矩阵，每 token 独立</td>
      <td style="text-align: left; padding: 6px 12px;">(m,)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">pergroup</td>
      <td style="text-align: left; padding: 6px 12px;">G</td>
      <td style="text-align: left; padding: 6px 12px;">左或右，k 轴分组</td>
      <td style="text-align: left; padding: 6px 12px;">左：(m, k/gs)；右：(k/gs, n)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">perblock</td>
      <td style="text-align: left; padding: 6px 12px;">B</td>
      <td style="text-align: left; padding: 6px 12px;">左或右，多轴分块</td>
      <td style="text-align: left; padding: 6px 12px;">左：(m/bs, k/bs)；右：(k/bs, n/bs)</td>
    </tr>
  </tbody>
</table>
</div>


选择量化粒度时，需要综合考虑精度要求、数据分布特征和硬件实现约束。以下逐一介绍五种粒度的原理，典型应用场景见第 4 节组合量化模式总结表。

### 3.1 T 量化（pertensor）

T 量化（pertensor）是最粗粒度的量化方式：整个 Tensor 的所有元素共享同一个 Scale，Scale shape 为 **(1,)**。由于所有元素使用统一的缩放因子，当 Tensor 内数据分布差异较大时，量化误差会显著增加。因此 T 量化适用于对精度要求不高、追求极致压缩比的场景。

<img src="./images/01.06/pertensor_quantization.png" alt="T量化原理" width="600px">


### 3.2 C 量化（perchannel）

C 量化（perchannel）针对右矩阵的每个输出 channel（n 轴方向）分配独立的 Scale，Scale shape 为 **(n,)**。由于神经网络权重矩阵的不同输出通道通常具有不同的数值分布，按 channel 独立量化能更好地保留各通道的动态范围，是大语言模型 weight 量化中较常用的方式。

<img src="./images/01.06/perchannel_quantization.png" alt="C量化原理" width="500px">


### 3.3 K 量化（pertoken）

K 量化（pertoken）针对左矩阵的每个 token（m 轴方向）分配独立的 Scale，Scale shape 为 **(m,)**。在 LLM 推理中，不同 token 的激活值分布差异显著，按 token 独立量化可以更精确地适应每个 token 的动态范围，是 activation 动态量化的核心方式。

<img src="./images/01.06/pertoken_quantization.png" alt="K量化原理" width="500px">


### 3.4 G 量化（pergroup）

G 量化（pergroup）在 reduce 轴 k 上按 **group size (gs)** 将元素分组，每组分配独立的 Scale。左矩阵 (m, k) 的 Scale shape 为 **(m, k/gs)**，右矩阵 (k, n) 的 Scale shape 为 **(k/gs, n)**。G 量化在精度与开销之间取得较好平衡，是 MX 量化等先进量化方案的基础。

<img src="./images/01.06/pergroup_quantization.png" alt="G量化原理" width="600px">


### 3.5 B 量化（perblock）

B 量化（perblock）在 m、k（左矩阵）或 k、n（右矩阵）两个轴上按 **block size (bs)** 进行二维分块，每个块拥有独立的 Scale。左矩阵 Scale shape 为 **(m/bs, k/bs)**，右矩阵为 **(k/bs, n/bs)**。B 量化提供了最细粒度的误差控制能力，但 Scale 存储开销也最大。

<img src="./images/01.06/perblock_quantization.png" alt="B量化原理" width="600px">


---

## 4. 常见组合量化模式

在实际算子开发中，左矩阵和右矩阵往往采用不同的量化粒度进行组合，以在精度与性能之间取得最优平衡。

**全量化**（左、右矩阵均量化）：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">简称</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">左矩阵</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">右矩阵</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">典型场景</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">T-C</td>
      <td style="text-align: left; padding: 6px 12px;">pertensor</td>
      <td style="text-align: left; padding: 6px 12px;">perchannel</td>
      <td style="text-align: left; padding: 6px 12px;">通用推理部署：左矩阵整 Tensor 共享单一 Scale，右矩阵 weight 按输出 channel 独立量化；实现简单、Scale 开销低，适用于小型模型全连接层 weight 整层统一量化或作为量化基线对比</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">K-C</td>
      <td style="text-align: left; padding: 6px 12px;">pertoken</td>
      <td style="text-align: left; padding: 6px 12px;">perchannel</td>
      <td style="text-align: left; padding: 6px 12px;">LLM 推理主流方案：左矩阵 activation 按 token 动态量化，适配不同 token 的 hidden state 分布变化；右矩阵 Linear 层 weight [K, N] 按列静态量化，兼顾精度与推理性能</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">G-B</td>
      <td style="text-align: left; padding: 6px 12px;">pergroup</td>
      <td style="text-align: left; padding: 6px 12px;">perblock</td>
      <td style="text-align: left; padding: 6px 12px;">混合粒度高精度方案：左矩阵在 k 轴按 group size 分组量化；右矩阵在 k、n 轴二维分块量化，适用于大模型 weight 按 k 轴分组且需块级精细误差控制的场景</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">G-G</td>
      <td style="text-align: left; padding: 6px 12px;">pergroup</td>
      <td style="text-align: left; padding: 6px 12px;">pergroup</td>
      <td style="text-align: left; padding: 6px 12px;">对称分组量化方案：左右矩阵均在 k 轴按 group size 分组量化，左 Scale shape 为 (m, k/gs)、右为 (k/gs, n)；精度与存储开销平衡较好，是大模型训推的主流低 bit 方案。第 5 节介绍的 MX 量化（MXFP8/MXFP4）即 OCP 标准化的 G-G 特例，group size 固定为 32</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">T-CG</td>
      <td style="text-align: left; padding: 6px 12px;">pertensor</td>
      <td style="text-align: left; padding: 6px 12px;">perchannel + pergroup</td>
      <td style="text-align: left; padding: 6px 12px;">扩展组合方案：左矩阵整 Tensor 量化降低开销；右矩阵同时采用 channel 与 group 两级 Scale，用于特定精度与压缩比的定制需求</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">B-B</td>
      <td style="text-align: left; padding: 6px 12px;">perblock</td>
      <td style="text-align: left; padding: 6px 12px;">perblock</td>
      <td style="text-align: left; padding: 6px 12px;">块级精细量化：左右矩阵均在 m-k 或 k-n 二维块级别独立 Scale，误差控制最细；适用于超大规模矩阵或对量化精度要求极高的场景</td>
    </tr>
  </tbody>
</table>
</div>

**伪量化**：仅对 weight 执行量化，activation 保持原精度参与计算，如 **C 量化**（perchannel）模式。

---

## 5. MX 量化模式实现介绍

如上表 **G-G** 组合所示，MX（Microscaling）量化是 G-G（pergroup-pergroup）量化模式的特例，由 OCP（Open Compute Project）标准化定义。其核心思想是在 k 轴方向以固定 group size 分组，每组通过独立的 Scale 动态调整缩放因子，以在 MXFP8、MXFP4 等低 bit 浮点格式下平衡精度与存储效率。

### 5.1 MX 量化参数规格

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">对比项</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">MXFP8</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">MXFP4</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">A/B 数据类型</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e4m3/float8_e5m2</td>
      <td style="text-align: left; padding: 6px 12px;">float4_e2m1/float4_e1m2</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">量化方式</td>
      <td style="text-align: left; padding: 6px 12px;">GroupSize=32 的 pergroup 量化</td>
      <td style="text-align: left; padding: 6px 12px;">GroupSize=32 的 pergroup 量化</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">ScaleA/B 数据类型</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e8m0</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e8m0</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">显存压缩比</td>
      <td style="text-align: left; padding: 6px 12px;">相比 FP16/BF16 减少约 50%</td>
      <td style="text-align: left; padding: 6px 12px;">相比 FP16/BF16 减少约 75%</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">典型应用场景</td>
      <td style="text-align: left; padding: 6px 12px;">模型训推，兼顾速度与精度</td>
      <td style="text-align: left; padding: 6px 12px;">模型推理，侧重极致显存效率</td>
    </tr>
  </tbody>
</table>
</div>

上表所列数据类型均遵循 `float{位数}_e{指数位}m{尾数位}` 命名约定：`e`（exponent）为指数位，决定数值可表示的动态范围；`m`（mantissa）为尾数位，决定数值的有效精度；通常还包含 1 位符号位，满足 `1 + e + m = 总位数`。

- `float8_e4m3` / `float8_e5m2`：MXFP8 场景下 A/B 矩阵的 8 bit 浮点格式。`e4m3` 为 1 符号 + 4 指数 + 3 尾数，精度与范围较为均衡；`e5m2` 指数位更多、尾数位更少，动态范围更大但精度更低。
- `float4_e2m1` / `float4_e1m2`：MXFP4 场景下 A/B 矩阵的 4 bit 浮点格式，显存占用约为 FP8 的一半。
- `float8_e8m0`：MX 量化场景中 ScaleA/ScaleB 专用格式，8 bit 全部用于指数、无尾数位，本质上只编码 2 的整数次幂缩放因子。每个 group（32 个元素）共享一个 scale。

以 MXFP8 量化场景，输入float8_e4m3为例，输入输出的具体规格如下：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">变量</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">描述</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Dtype</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Layout</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Shape</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">a</td>
      <td style="text-align: left; padding: 6px 12px;">输入左矩阵</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e4m3/float8_e5m2</td>
      <td style="text-align: left; padding: 6px 12px;">ND</td>
      <td style="text-align: left; padding: 6px 12px;">(m, k)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">b</td>
      <td style="text-align: left; padding: 6px 12px;">输入右矩阵</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e4m3/float8_e5m2</td>
      <td style="text-align: left; padding: 6px 12px;">ND</td>
      <td style="text-align: left; padding: 6px 12px;">(n, k)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">scaleA</td>
      <td style="text-align: left; padding: 6px 12px;">左矩阵量化参数</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e8m0</td>
      <td style="text-align: left; padding: 6px 12px;">ND</td>
      <td style="text-align: left; padding: 6px 12px;">(m, ceil(k/64), 2)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">scaleB</td>
      <td style="text-align: left; padding: 6px 12px;">右矩阵量化参数</td>
      <td style="text-align: left; padding: 6px 12px;">float8_e8m0</td>
      <td style="text-align: left; padding: 6px 12px;">ND</td>
      <td style="text-align: left; padding: 6px 12px;">(n, ceil(k/64), 2)</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">c</td>
      <td style="text-align: left; padding: 6px 12px;">输出矩阵</td>
      <td style="text-align: left; padding: 6px 12px;">bfloat16</td>
      <td style="text-align: left; padding: 6px 12px;">ND</td>
      <td style="text-align: left; padding: 6px 12px;">(m, n)</td>
    </tr>
  </tbody>
</table>
</div>

计算公式为：

$$
c_{i, j} = \sum^{ceil(K/G)-1}_{g=0}\left(scaleA_{i, g} \cdot scaleB_{g, j} \cdot \sum^{G-1}_{k'=0} (a_{i, gG+k'} \cdot b_{gG + k', j}) \right)
$$

其中 G 为 group size，在 MX 量化场景固定取值为32。

### 5.2 MX 量化算子实现原理

与传统非量化 MatMul 算子相比，MX 量化场景新增了输入变量 `scaleA` 和 `scaleB`，需要将其搬运至 L1 缓冲区，再搬运至 `L0A_MX` 和 `L0B_MX` 独立缓冲区。通过约束 `L0A`、`L0B`、`L0A_MX`、`L0B_MX` 中 Tensor 的地址映射关系，芯片的 `MMAD` 指令支持自动完成 MXFP8 矩阵乘，并将 Float32 结果写入 `L0C` 缓冲区。通过设置 `Fixpipe` 指令的量化模式，输出预期的数据类型结果。因此算子实现仅依赖 CUBE 核，不涉及 MIX 场景。

MX 量化场景矩阵乘执行时的完整数据搬运流程如下图所示：

<img src="./images/01.06/mx/mx-quant-matmul-data-movement-overview.png" alt="MX量化矩阵乘数据搬运总览" width="800px">


#### Tensor a 的搬运路径

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">缓冲区变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Shape 排布变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Layout 变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所属流水</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所用指令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">GM → L1</td>
      <td style="text-align: left; padding: 6px 12px;">(m, k) → (ceil(kL1/k0), ceil(mL1/m0), m0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">ND → Nz</td>
      <td style="text-align: left; padding: 6px 12px;">MTE2</td>
      <td style="text-align: left; padding: 6px 12px;">DataCopy with ND2NZ</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">L1 → L0A</td>
      <td style="text-align: left; padding: 6px 12px;">(ceil(kL1/k0), ceil(mL1/m0), m0, k0) → (ceil(baseK/k0), ceil(baseM/m0), m0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">Nz → Nz</td>
      <td style="text-align: left; padding: 6px 12px;">MTE1</td>
      <td style="text-align: left; padding: 6px 12px;">LoadData with Load2D</td>
    </tr>
  </tbody>
</table>
</div>

<img src="./images/01.06/mx/mx-quant-tensor-a-gm-l1-l0-path.png" alt="Tensor a 搬运路径" width="900px">

> MXFP8 场景下 K0 = 32，表示沿 K 轴按 32 对齐后的最小分块进行组织。

> 注：layout 有分形存储思想时，外层分形排布大写，内层 tile 元素排布小写，故有Nz（大 N，小 z）格式。（下文同理）

#### Tensor b 的搬运路径

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">缓冲区变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Shape 排布变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Layout 变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所属流水</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所用指令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">GM → L1</td>
      <td style="text-align: left; padding: 6px 12px;">(n, k) → (ceil(kL1/k0), ceil(nL1/n0), n0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">DN → Zn</td>
      <td style="text-align: left; padding: 6px 12px;">MTE2</td>
      <td style="text-align: left; padding: 6px 12px;">DataCopy with DN2ZN</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">L1 → L0B</td>
      <td style="text-align: left; padding: 6px 12px;">(ceil(kL1/k0), ceil(nL1/n0), n0, k0) → (ceil(baseK/k0), ceil(baseN/n0), n0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">Zn → Zn</td>
      <td style="text-align: left; padding: 6px 12px;">MTE1</td>
      <td style="text-align: left; padding: 6px 12px;">LoadData with Load2D</td>
    </tr>
  </tbody>
</table>
</div>


<img src="./images/01.06/mx/mx-quant-tensor-b-gm-l1-l0-path.png" alt="Tensor b 搬运路径" width="900px">

#### Tensor scaleA 的搬运路径

由于 MX 量化采用 GroupSize=32 的 pergroup 量化，Scale 在 K 方向的大小是输入矩阵的 1/32。

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">缓冲区变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Shape 排布变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Layout 变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所属流水</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所用指令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">GM → L1</td>
      <td style="text-align: left; padding: 6px 12px;">(m, ceil(k/64), 2) → (ceil(mL1/m0), ceil(ceil(kL1/32)/k0), m0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">ND → Zz</td>
      <td style="text-align: left; padding: 6px 12px;">MTE2</td>
      <td style="text-align: left; padding: 6px 12px;">DataCopy with ND2NZ</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">L1 → L0A_MX</td>
      <td style="text-align: left; padding: 6px 12px;">(ceil(mL1/m0), ceil(ceil(kL1/32)/k0), m0, k0) → (ceil(baseM/m0), ceil(ceil(baseK/32)/k0), m0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">Zz → Zz</td>
      <td style="text-align: left; padding: 6px 12px;">MTE1</td>
      <td style="text-align: left; padding: 6px 12px;">LoadData with Load2D_MX</td>
    </tr>
  </tbody>
</table>
</div>


<img src="./images/01.06/mx/mx-quant-scalea-gm-l1-l0-mx-path.png" alt="Tensor scaleA 搬运路径" width="900px">

#### Tensor scaleB 的搬运路径

<div style="width: 100%;">
<table style="border-collapse: collapse; width: auto; max-width: 100%; margin-left: 0 !important; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left; padding: 6px 12px 6px 0; border-bottom: 1px solid #ddd;">缓冲区变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Shape 排布变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">Layout 变化</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所属流水</th>
      <th style="text-align: left; padding: 6px 12px; border-bottom: 1px solid #ddd;">所用指令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">GM → L1</td>
      <td style="text-align: left; padding: 6px 12px;">(n, ceil(k/64), 2) → (ceil(nL1/n0), ceil(ceil(kL1/32)/k0), n0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">DN → Nn</td>
      <td style="text-align: left; padding: 6px 12px;">MTE2</td>
      <td style="text-align: left; padding: 6px 12px;">DataCopy with DN2NZ</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 6px 12px 6px 0;">L1 → L0B_MX</td>
      <td style="text-align: left; padding: 6px 12px;">(ceil(nL1/n0), ceil(ceil(kL1/32)/k0), n0, k0) → (ceil(baseN/n0), ceil(ceil(baseK/32)/k0), n0, k0)</td>
      <td style="text-align: left; padding: 6px 12px;">Nn → Nn</td>
      <td style="text-align: left; padding: 6px 12px;">MTE1</td>
      <td style="text-align: left; padding: 6px 12px;">LoadData with Load2D_MX</td>
    </tr>
  </tbody>
</table>
</div>


<img src="./images/01.06/mx/mx-quant-scaleb-gm-l1-l0-mx-path.png" alt="Tensor scaleB 搬运路径" width="900px">

### 5.3 实现约束要点

MX 量化算子实现需关注以下关键约束：

1. **K 维度对齐**：MMAD 指令要求输入矩阵在 K 方向长度为 64 的倍数，因此 **baseK 必须是 64 的整数倍**
2. **指令约束**：`MMAD` 指令需关闭 `gemv` 功能
3. **K 维度补零**：当 K 轴非 64 对齐时，需根据输入排布选择自动补零或手动补零策略

---

## 课后练习

1. **（判断题）** MatMul 计算中的量化操作是指在 Cube 计算完成后，通过 Scale 将低 bit 结果转换回高 bit，以保证与全精度计算近似等价。

2. **（判断题）** 动态量化因为需要在线生成量化参数，在推理场景下通常比静态量化具有更好的性能。

3. **（单选题）** 右矩阵 shape 为 (k, n)，采用 perchannel 量化时，Scale 的 shape 应为：
   - A. (1,)　B. (m,)　C. (n,)　D. (k/gs, n)

4. **（单选题）** LLM 推理中，左矩阵 activation 采用动态量化且 Scale shape 为 (m,)，对应的量化粒度为：
   - A. T 量化（pertensor）　B. C 量化（perchannel）　C. K 量化（pertoken）　D. B 量化（perblock）

5. **（单选题）** MX 量化本质是哪种量化模式的特例？
   - A. T-C 量化　B. G-G 量化　C. B-B 量化　D. C 量化

6. **（多选题）** 以下关于 pergroup 量化的描述，正确的有：
   - A. 在 reduce 轴 k 上按 group size 分组
   - B. 左矩阵 (m,k) 的 Scale shape 为 (m, k/gs)
   - C. 右矩阵 (k,n) 的 Scale shape 为 (k/gs, n)
   - D. group size 固定为 16

7. **（填空题）** 左矩阵 shape 为 (m, k)、右矩阵 shape 为 (k, n)，采用 G-G 组合量化（group size = gs）时，ScaleA 的逻辑 shape 为 ______，ScaleB 的逻辑 shape 为 ______。

8. **（填空题）** MX 量化中，Scale 的数据类型为 ______，group size 为 ______。

---

## 参考答案

In [ ]:
!cat ./answer/01.06_answer.txt